# Graphons and GNNs for Physical Diffusion Prediction

This notebook presents multiple graphon generators, including a Perlin noise generator, and demonstrates how to sample graphs from them. For each graphon, the corresponding image and a few sampled graphs are shown. Finally, a simple Graph Neural Network (GNN) is built to predict a physical phenomenon: the diffusion of a substance on the network.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from typing import Callable, List, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, DataLoader
from torch_geometric.nn import GCNConv

try:
    from noise import pnoise2
except ImportError:
    raise ImportError('Please install the noise package: pip install noise')
 
np.random.seed(42)
torch.manual_seed(42)

## Graphon Generators

In [ ]:
from logic.graphon_generator import generate_constant_graphon, generate_min_graphon, generate_perlin_graphon, generate_piecewise_graphon

graphon_generators = [
    ("Constant Graphon (p=0.5)", generate_constant_graphon(0.5)),
    ("Piecewise Graphon (k=4)", generate_piecewise_graphon(4)),
    ("Min Graphon", generate_min_graphon()),
    ("Perlin Noise Graphon", generate_perlin_graphon())
]

## Physical processes over graphs

### Diffusion Process

In [ ]:
import matplotlib.animation as animation
from matplotlib import cm
from IPython.display import HTML

from logic.graphon_generator import sample_graph

def simulate_diffusion_step(A: np.ndarray, features: np.ndarray, diffusion_rate: float = 0.1) -> np.ndarray:
    d = np.array(A.sum(axis=1)).flatten()
    D = np.diag(d)
    L = D - A
    new_features = features - diffusion_rate * (L @ features)
    return new_features

def animate_diffusion(G: nx.Graph, initial_features: np.ndarray, steps: int = 20, diffusion_rate: float = 0.1) -> HTML:
    pos = nx.spring_layout(G, seed=42)
    A = nx.adjacency_matrix(G).todense()
    features = initial_features.copy()
    
    fig, ax = plt.subplots(figsize=(5, 5))
    cmap = cm.viridis
    nodes = nx.draw_networkx_nodes(G, pos, node_size=300,
                                   node_color=features.flatten(), cmap=cmap, ax=ax, vmin=0, vmax=1)
    nx.draw_networkx_edges(G, pos, ax=ax)
    ax.set_title("Diffusion Process Animation")
    ax.axis("off")
    
    def update(frame: int) -> None:
        nonlocal features
        features = simulate_diffusion_step(A, features, diffusion_rate)
        nodes.set_array(features.flatten())
        ax.set_title(f"Diffusion Step {frame+1}")
        return nodes,
    
    ani = animation.FuncAnimation(fig, update, frames=steps, interval=500, blit=False)
    plt.close(fig)
    return HTML(ani.to_jshtml())

# Demo: Animate diffusion on a sample graph
sample_G = sample_graph(interp_graphon(generate_min_graphon()), n=30)
initial_feat = np.random.rand(30, 1)
animate_diffusion(sample_G, initial_feat, steps=25)